# Sistema de Recomendação — LH Nautical

## Produto de referência: Motor de Popa 1949

Este notebook implementa um motor de recomendação do tipo **“Quem comprou isso, também levou...”** baseado na similaridade de comportamento de compra dos clientes.

### Regras do desafio

1. Matriz **Usuário × Produto**:
   - linhas: `customer_id`;
   - colunas: `product_id`;
   - valor `1` se o cliente comprou ao menos uma vez;
   - valor `0` caso contrário;
   - quantidade comprada é ignorada.
2. Similaridade de **Cosseno** entre os vetores dos produtos.
3. Ranking dos **5 produtos mais similares** ao `Motor de Popa 1949`, excluindo o próprio produto.

### Regra de negócio adotada

Somente pedidos com status `paid` ou `confirmed` são considerados como compras válidas. Pedidos `cancelled` e `draft` são ignorados.


In [4]:
from pathlib import Path
import sys
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

# Permite importar o código do componente dentro do container.
sys.path.insert(0, "/workspace/recommendation/src")

from dataset import load_raw_data, build_interaction_dataset
from similarity import build_user_product_matrix, build_product_similarity
from recommender import get_product_id, rank_similar_products

TARGET_PRODUCT = "Motor de Popa 1949"
TOP_N = 5

DATA_DIR = Path("/workspace/data/raw")
PROCESSED_PATH = Path(
    "/workspace/data/processed/recommendation_interactions.csv"
)

# Fallback para execução local fora do Docker.
if not DATA_DIR.exists():
    DATA_DIR = Path("data/raw")

print("Diretório de dados:", DATA_DIR.resolve())


Diretório de dados: /workspace/data/raw


## 1. Carregar os quatro datasets

A leitura e os joins são encapsulados em `recommendation/src/dataset.py`, mantendo o notebook focado na análise e permitindo reutilização do código fora do Jupyter.


In [5]:
raw = load_raw_data(DATA_DIR)

products = raw["products"]
variants = raw["variants"]
orders = raw["orders"]
order_items = raw["order_items"]

for name, df in raw.items():
    print(f"{name}: {df.shape[0]:,} linhas x {df.shape[1]} colunas")


products: 500 linhas x 10 colunas
variants: 1,009 linhas x 12 colunas
orders: 48,998 linhas x 13 colunas
order_items: 147,320 linhas x 8 colunas


## 2. Identificar o produto de referência e suas variantes

A recomendação é feita no nível de **produto**, portanto as diferentes variantes/SKUs do `Motor de Popa 1949` são consolidadas no mesmo `product_id`.


In [6]:
target_products = products[
    products["name"].astype(str).str.strip().str.casefold()
    == TARGET_PRODUCT.casefold()
].copy()

if target_products.empty:
    raise ValueError(f"Produto não encontrado: {TARGET_PRODUCT}")

display(target_products[["id", "name", "is_active"]])

target_product_ids = target_products["id"].tolist()

target_variants = variants[
    variants["product_id"].isin(target_product_ids)
][["id", "product_id", "sku", "is_active"]].copy()

print("Variantes do produto:")
display(target_variants)


,id,name,is_active
179,180,Motor de Popa 1949,True


Variantes do produto:


,id,product_id,sku,is_active
363,364,180,LHN-132290,True
364,365,180,LHN-830083,True
365,366,180,LHN-581742,True


## 3. Construir o dataset de interações

Cada combinação `customer_id × product_id` aparece apenas uma vez. Mesmo que o cliente tenha comprado o mesmo produto várias vezes ou em quantidades maiores, o valor da interação continua sendo **1**.


In [7]:
interactions = build_interaction_dataset(
    products=products,
    variants=variants,
    orders=orders,
    order_items=order_items,
)

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
interactions.to_csv(PROCESSED_PATH, index=False)

print(f"Interações únicas: {len(interactions):,}")
print(f"Clientes: {interactions['customer_id'].nunique():,}")
print(f"Produtos: {interactions['product_id'].nunique():,}")
print(f"Dataset salvo em: {PROCESSED_PATH}")

display(interactions.head())


Interações únicas: 116,854
Clientes: 2,000
Produtos: 500
Dataset salvo em: /workspace/data/processed/recommendation_interactions.csv


,customer_id,product_id,product_name,purchased
0,1,2,Vela Mestra 3870,1
1,1,21,Âncora Bruce 6850,1
2,1,22,Vela Mestra 7039,1
3,1,24,Colete Salva-Vidas 3398,1
4,1,47,Motor de Popa 5561,1


## 4. Criar a matriz Usuário × Produto

A matriz abaixo segue exatamente a especificação da tarefa:

- linhas = clientes;
- colunas = produtos;
- `1` = comprou ao menos uma vez;
- `0` = não comprou.


In [8]:
user_product_matrix = build_user_product_matrix(interactions)

print("Dimensão da matriz Usuário × Produto:")
print(user_product_matrix.shape)

unique_values = set(user_product_matrix.stack().unique())
print("Valores encontrados na matriz:", unique_values)

assert unique_values.issubset({0, 1})

display(user_product_matrix.iloc[:10, :10])


Dimensão da matriz Usuário × Produto:
(2000, 500)
Valores encontrados na matriz: {np.int8(0), np.int8(1)}


product_id,1,2,3,4,5,6,7,8,9,10
customer_id,,,,,,,,,,
1,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,1,0,0,1,0,0
5,0,0,1,0,0,1,0,0,1,0
6,1,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0
8,0,0,1,0,0,0,0,0,0,1
9,1,1,0,1,0,0,0,0,0,1


## 5. Representar produtos por seus vetores de clientes

Para comparar produtos, a matriz é transposta. Assim, cada linha passa a representar um produto e cada coluna um cliente.


In [ ]:
product_user_matrix = user_product_matrix.T

print("Dimensão Produto × Usuário:")
print(product_user_matrix.shape)

display(product_user_matrix.iloc[:10, :10])


## 6. Calcular Similaridade de Cosseno Produto × Produto

A similaridade de cosseno compara o padrão de clientes que compraram cada item. Quanto mais próximo de `1`, maior a sobreposição relativa entre os vetores de compradores.


In [9]:
similarity_matrix = build_product_similarity(
    user_product_matrix
)

print("Dimensão da matriz Produto × Produto:")
print(similarity_matrix.shape)

display(similarity_matrix.iloc[:10, :10])


Dimensão da matriz Produto × Produto:
(500, 500)


product_id,1,2,3,4,5,6,7,8,9,10
product_id,,,,,,,,,,
1,1.000000,0.193265,0.146990,0.133071,0.200044,0.164224,0.139777,0.095806,0.150514,0.101290
2,0.193265,1.000000,0.143860,0.151775,0.206317,0.173275,0.143314,0.111096,0.176823,0.086222
3,0.146990,0.143860,1.000000,0.114027,0.146683,0.144072,0.144488,0.103699,0.168658,0.069435
4,0.133071,0.151775,0.114027,1.000000,0.166077,0.107511,0.126215,0.050996,0.117093,0.102438
5,0.200044,0.206317,0.146683,0.166077,1.000000,0.167692,0.125737,0.133871,0.189160,0.067229
6,0.164224,0.173275,0.144072,0.107511,0.167692,1.000000,0.170227,0.080915,0.174180,0.086349
7,0.139777,0.143314,0.144488,0.126215,0.125737,0.170227,1.000000,0.091827,0.177901,0.109521
8,0.095806,0.111096,0.103699,0.050996,0.133871,0.080915,0.091827,1.000000,0.094840,0.043669
9,0.150514,0.176823,0.168658,0.117093,0.189160,0.174180,0.177901,0.094840,1.000000,0.105282


## 7. Localizar o Motor de Popa 1949


In [10]:
product_catalog = (
    interactions[["product_id", "product_name"]]
    .drop_duplicates()
    .sort_values("product_id")
)

target_product_id = get_product_id(
    product_catalog,
    TARGET_PRODUCT,
)

print("Produto:", TARGET_PRODUCT)
print("product_id:", target_product_id)


Produto: Motor de Popa 1949
product_id: 180


## 8. Ranking dos 5 produtos mais similares

O próprio `Motor de Popa 1949` é removido antes da ordenação final.


In [11]:
ranking = rank_similar_products(
    target_product_id=target_product_id,
    similarity_matrix=similarity_matrix,
    product_catalog=product_catalog,
    top_n=TOP_N,
)

ranking["similarity"] = ranking["similarity"].round(6)

display(ranking)


,rank,product_id,product_name,similarity
0,1,75,Vela Mestra 1913,0.245200
1,2,295,Cabo Náutico 2105,0.229962
2,3,311,GPS Plotter 2249,0.214818
3,4,308,Motor de Popa 1540,0.212121
4,5,2,Vela Mestra 3870,0.208824


## 9. Resultado esperado com os arquivos fornecidos

Utilizando os CSVs disponibilizados e considerando apenas pedidos `paid` e `confirmed`, o ranking esperado é:

| Rank | Produto | Similaridade |
|---:|---|---:|
| 1 | Vela Mestra 1913 | 0,245200 |
| 2 | Cabo Náutico 2105 | 0,229962 |
| 3 | GPS Plotter 2249 | 0,214818 |
| 4 | Motor de Popa 1540 | 0,212121 |
| 5 | Vela Mestra 3870 | 0,208824 |

Portanto, pelo critério solicitado de similaridade de comportamento de compra, **Vela Mestra 1913** é o primeiro produto do ranking para acompanhar `Motor de Popa 1949`.


## 10. Interpretação e limitação

### Como o recomendador funciona?

O modelo é **item-based**: dois produtos são considerados semelhantes quando são comprados por conjuntos semelhantes de clientes. Não são utilizadas características do produto, quantidade comprada ou conteúdo textual.

### Uma limitação

A similaridade mede **coocorrência de comportamento**, e não necessariamente complementaridade comercial. Dois produtos podem ter alta similaridade porque atendem ao mesmo perfil de cliente, sem que façam sentido como compra conjunta na mesma ocasião. Além disso, produtos novos ou pouco comprados sofrem com o problema de *cold start*.
